# 第十七课:Top-P(核采样)

讲义:[17-top-p.md](17-top-p.md)。

In [ ]:
import numpy as np

def softmax(x):
    m = np.max(x)
    e = np.exp(x - m)
    return e / e.sum()

def top_p(logits, p):
    order = np.argsort(-logits)
    cum = np.cumsum(softmax(logits)[order])
    cutoff = np.searchsorted(cum, p) + 1
    keep = order[:cutoff]
    mask = np.full_like(logits, -np.inf)
    mask[keep] = logits[keep]
    return mask, keep, cum

logits = np.array([1.0, 4.0, 2.0, 0.0])
mask, keep, cum = top_p(logits, 0.85)
print('排序概率:', np.round(softmax(logits)[np.argsort(-logits)], 4))
print('累积和  :', np.round(cum, 4))
print('保留 idx:', keep, ' 概率:', np.round(softmax(mask), 4))

In [ ]:
# 自适应:同一 p,尖/平分布保留人数不同
sharp = np.array([10., 8, 0, 0, 0, 0])      # 尖分布
flat  = np.array([1., 0.9, 0.8, 0.7, 0.6, 0.5])   # 平分布
for name, lg in [('尖', sharp), ('平', flat)]:
    _, keep, cum = top_p(lg, 0.9)
    print(f'{name}分布 p=0.9 → 保留 {len(keep)} 个词,  累积={np.round(cum, 3)}')

In [ ]:
# TODO(C3): 完整解码器
def sample(logits, rng, T=0.7, p=0.9):
    # 温度 → 截断 → 采样(你的代码)
    ...

# TODO(D2): 边界 bug 测试:两词分布 p0 恰好等于 p_th
# lg = np.array([np.log(0.9/(1-0.9)), 0.0]) 构造 p0=0.9, 检查 top_p(lg, 0.9) 保留几个

# TODO(E1): p 从 0.5 到 1.0 扫描,画两种分布的保留词数曲线